# trillion-serving

The numpy and planner cells of the post, in its order, with no outputs. Run them from the repo root after the setup in the README. Later cells use names that earlier cells made.

In [ ]:
import sys
sys.path.insert(0, "trillion")                       # config.py and capacity.py import each other flat

## Weight Formats and Model Bytes

In [ ]:
import numpy as np

FP4 = np.array([0, 0.5, 1, 1.5, 2, 3, 4, 6])        # every size a 4-bit E2M1 number can hold

def mxfp4(block):
    """32 weights share one power-of-two scale, then each keeps a 4-bit code."""
    scale = 2.0 ** (np.floor(np.log2(np.abs(block).max())) - 2)   # the biggest weight divides to 4 to 8, then clips to 6
    size = np.abs(FP4[:, None] - np.abs(block) / scale).argmin(axis=0)
    return scale, np.sign(block) * FP4[size]

w = np.random.default_rng(2).normal(0, 0.02, 32).round(4)   # one block of 32 weights
scale, codes = mxfp4(w)
print("weights", w[:6])
print("codes  ", codes[:6], "  scale", scale)
print("back   ", (codes * scale)[:6])
print(f"{(32 * 4 + 8) / 32} bits a weight, mean error {np.abs(codes * scale - w).mean():.5f}")

## Matrix Products and FLOP Counting

In [ ]:
x = np.array([2, 1, 0, 3])                            # one token, four numbers
W = np.array([[1, 0, 2], [3, 1, 0], [0, 2, 1], [1, 1, 1]])
print("y =", x @ W, "  FLOPs:", 2 * W.size)           # one multiply and one add per weight
print(f"one token of Kimi K3: {2 * 104.2e9 / 1e9:.1f} GFLOP")
for batch in (1, 64, 312, 1024):                      # tokens that share one read of an FP8 weight
    print(f"batch {batch:>4}: {2 * batch:>5} FLOPs for every byte read")
print("the GPU stops waiting on memory at", 5e15 / 8e12, "FLOPs a byte")

## Mixture of Experts Routing

In [ ]:
logits = np.array([1.2, -0.3, 2.9, 0.4, 2.1, -1.0, 0.8, 1.7])   # one token against 8 experts
scores = 1 / (1 + np.exp(-logits))                    # a sigmoid scores each expert on its own
pick = np.argsort(scores)[::-1][:2]                   # keep the best 2
weight = scores[pick] / scores[pick].sum()            # and mix their outputs in these shares
print("scores ", scores.round(3))
print("experts", pick, " weights", weight.round(3))
print(f"GPUs one token visits: {72 * (1 - (71 / 72) ** 16):.1f} of 72")

## Attention and the KV Cache

In [ ]:
q = np.array([1.0, 0.5])                              # the newest token asks a question
K = np.array([[1.0, 0.0], [0.2, 1.0], [0.9, 0.4]])    # the keys of the three tokens before it
V = np.array([[1.0, 2.0], [3.0, 0.0], [0.0, 1.0]])    # and their values
s = K @ q / np.sqrt(2)                                # how well each key answers
a = np.exp(s) / np.exp(s).sum()                       # softmax: shares that add up to 1
print("scores", s.round(3), " shares", a.round(3), " output", (a @ V).round(3))

In [ ]:
rng = np.random.default_rng(0)
q, k = rng.normal(size=(2, 100_000, 192))            # random query and key pairs, 192 numbers each
dots = (q * k).sum(axis=1)
spread, scaled = dots.std(), (dots / np.sqrt(192)).std()
print(f"spread of q.k: {spread:.1f}   divided by sqrt(192): {scaled:.2f}")
for d in (1, 192):                                    # four scores through softmax, raw then scaled
    e = np.exp(dots[:4] / np.sqrt(d))
    print(f"softmax, divided by sqrt({d}):", (e / e.sum()).round(3))

## Latent Compression in MLA

In [ ]:
values = 96 * (192 + 128)                             # numbers a plain cache keeps per token per layer
latent = 512 + 64                                     # MLA keeps the latent and one shared key slot
print(f"{values:,} against {latent}: {values / latent:.1f}x fewer numbers per layer")
print(f"over all 93 layers: {values * 93 / (latent * 24):.1f}x fewer, {latent * 24:,} B a token in FP8")

## Recurrent State in KDA

In [ ]:
S = np.zeros((2, 2))                                  # one head's state, d_k rows by d_v columns
steps = [([1, 0], [2, 1], [1.0, 1.0], 1.0),           # key, value, decay per key channel, write strength
         ([0, 1], [1, 3], [0.9, 0.5], 1.0),
         ([0.6, 0.8], [0, 2], [0.9, 0.9], 0.5)]
for k, v, alpha, beta in steps:
    k, v = np.array(k), np.array(v)
    S = np.array(alpha)[:, None] * S                  # forget a little, per channel
    S = S + beta * np.outer(k, v - S.T @ k)           # write only what the state still gets wrong
    print(S.round(3).tolist())
print("read with the last key:", (S.T @ np.array([0.6, 0.8])).round(3), "  size:", S.shape)

In [ ]:
from capacity import request_bytes
for tokens in (2_048, 8_192, 32_768, 262_144, 1_048_576):
    state, latent = request_bytes(tokens)
    share = state / (state + latent)
    print(f"{tokens:>9,} tokens  state {state / 1e6:5.1f} MB  latent {latent / 1e6:8,.1f} MB  {share:4.0%} state")

## Rack Hardware and Bandwidth

In [ ]:
links = {"HBM3E": 8e12, "NVLink": 900e9, "network port": 100e9}   # bytes a second, one GPU, one way
for name, speed in links.items():
    print(f"{name:<13} 1 GB in {1e9 / speed * 1e3:6.3f} ms   one KDA state in {449.4e6 / speed * 1e3:5.2f} ms")

## Weights and Experts Across a Rack

In [ ]:
import config as cfg
from capacity import weights
static = cfg.MEM_FRACTION * cfg.HBM
for name, size in weights().items():
    print(f"{name:<32}weights {size / 1e9:6.1f} GB   left for requests {(static - size) / 1e9:6.1f} GB")

## Decode Step Budget

In [ ]:
from capacity import best_layout, decode_step
for tokens in (2_048, 8_192, 32_768, 262_144, 1_048_576):
    tp, rack = best_layout(tokens)
    hbm, flops, wire = decode_step(tokens, tp, rack * tp / cfg.GPUS)
    print(f"{tokens:>9,} tp{tp}  HBM {hbm * 1e3:4.1f} ms  FLOPs {flops * 1e3:4.1f} ms  wire {wire * 1e3:4.1f} ms")

## Splitting One Request Across GPUs

In [ ]:
s = np.array([0.9, 2.1, -0.4, 1.3])                   # one query's scores over four cached tokens
v = np.array([1.0, 4.0, 2.0, 3.0])                    # their values, one number each
full = np.exp(s) @ v / np.exp(s).sum()

def partial(s, v):
    """What one GPU returns for its half: its largest score, its sum of exponents and its answer."""
    m = s.max()
    e = np.exp(s - m)
    return m, e.sum(), e @ v / e.sum()

(m1, l1, o1), (m2, l2, o2) = partial(s[:2], v[:2]), partial(s[2:], v[2:])
m = max(m1, m2)
w1, w2 = l1 * np.exp(m1 - m), l2 * np.exp(m2 - m)    # both halves rescaled to one shared max
print(f"one GPU {full:.4f}   two GPUs merged {(w1 * o1 + w2 * o2) / (w1 + w2):.4f}")

## Attention Layout Sweep

In [ ]:
from capacity import best_layout, per_rack
for tokens in (2_048, 8_192, 32_768, 131_072, 262_144, 1_048_576):
    row = "  ".join(f"tp{tp} {per_rack(tokens, tp):>6,}" for tp in (1, 2, 4, 8))
    print(f"{tokens:>9,}  {row}  best tp{best_layout(tokens)[0]}")

In [ ]:
tp, held = best_layout(262_144)
print(f"tp{tp}: {held:,} per rack, {held / cfg.GPUS:.1f} per GPU, {100_000 / held:.1f} racks for 100,000")

In [ ]:
from capacity import wire_bytes
tp, rack = best_layout(3_048)
sent = rack * tp / cfg.GPUS * wire_bytes(tp)
print(f"tp{tp}: {sent / 1e9:.2f} GB per GPU each step, {sent / cfg.NVLINK * 1e3:.1f} ms on NVLink, {sent / cfg.NIC * 1e3:.1f} ms on the NIC")

## Disaggregated Serving and Routing

In [ ]:
import hashlib

racks = [f"http://prefill-{i}:30000" for i in range(13)]
ring = sorted((int(hashlib.sha1(f"{url}#{v}".encode()).hexdigest(), 16), url) for url in racks for v in range(64))
share = dict.fromkeys(racks, 0.0)
for (before, _), (point, url) in zip([ring[-1]] + ring[:-1], ring):   # a key belongs to the next point clockwise
    share[url] += (point - before) % 2**160 / 2**160
print(f"each rack holds {min(share.values()):.1%} to {max(share.values()):.1%} of sessions, against {1 / 13:.1%}")
print(f"prefill-6 holds {share['http://prefill-6:30000']:.1%}, and only those sessions move when it fails")

## Fleet Sizing With Little's Law

In [ ]:
from capacity import pools, size_cluster
plan = size_cluster()
for name, (tp, racks, step, arrivals, gpus) in plan.items():
    print(f"{name:<6} tp{tp}  decode racks {racks:4.2f}  step {step * 1e3:4.1f} ms  {arrivals:7,.1f} req/s  prefill GPUs {gpus:5.1f}")
print(pools(plan))

In [ ]:
for busy in (0.5, 0.75, 0.9, 0.95):                  # the share of time a prefill rack is busy
    print(f"{busy:.0%} busy: a new prompt waits {busy / (1 - busy):4.1f} service times on average")